# Notebook 07 — RQ1: Player Role Distribution Analysis ([#118](https://github.com/UBC-MDS/UBC-MDS-Soccer-Capstone-2026/issues/118))
 
**Research question (RQ1):** How does player role distribution differ across competitions and positions?

**Inputs (Databricks → BigQuery foreign catalog):**
- `dbt_intermediate.int_player_season_stats` — player-season per-90 metrics (270+ minutes)
- `analytics.cluster_assignments` — 5 archetypes from `src/ml/cluster.py` (13-feature PCA + K-Means)
- `dbt_intermediate.int_player_match_stats` + `matches` — primary `position_name` per player-season

**Scope:** Top-five European leagues (La Liga, EPL, Bundesliga, Serie A, Ligue 1); positions grouped as GK, CB, FB, CM, AM, FW.

**Key questions:**
1. Does **position** predict cluster membership?
2. Does **competition** affect role (cluster) distribution patterns?

**Outputs (for final report § RQ1):** Summary tables, chi-square / Cramér's V statistics, and heatmaps.

---
## 0. Setup

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="muted")

# Databricks foreign catalog → BigQuery (see docs/databricks_bigquery_setup.md)
BQ_CATALOG = os.environ.get("BQ_CATALOG", "bq_raw_statsbomb_sa_catalog")
INTERMEDIATE = f"{BQ_CATALOG}.dbt_intermediate"
INT_SEASON = f"{INTERMEDIATE}.int_player_season_stats"
INT_MATCH = f"{INTERMEDIATE}.int_player_match_stats"
CLUSTER_TABLE = f"{BQ_CATALOG}.analytics.cluster_assignments"
MATCHES = f"{BQ_CATALOG}.raw_statsbomb.matches"

# Top-five leagues — filter by name (StatsBomb competition_id varies by extract)
TOP5_RAW_NAMES = [
    "La Liga",
    "Premier League",
    "English Premier League",
    "1. Bundesliga",
    "Bundesliga",
    "Serie A",
    "Ligue 1",
]
NAME_TO_LABEL = {
    "La Liga": "La Liga",
    "Premier League": "EPL",
    "English Premier League": "EPL",
    "1. Bundesliga": "Bundesliga",
    "Bundesliga": "Bundesliga",
    "Serie A": "Serie A",
    "Ligue 1": "Ligue 1",
}
COMP_LABEL_ORDER = ["La Liga", "EPL", "Bundesliga", "Serie A", "Ligue 1"]
POSITION_ORDER = ["GK", "CB", "FB", "CM", "AM", "FW"]

print(f"Intermediate:  {INTERMEDIATE}")
print(f"Season stats:  {INT_SEASON}")
print(f"Clusters:      {CLUSTER_TABLE}")
print("Setup complete.")

---
## 1. Position groups (StatsBomb → GK/CB/FB/CM/AM/FW)

StatsBomb `lineups.position_name` has ~25 values. We map each to a coarse tactical group for RQ1.

In [ ]:
POSITION_MAP = {
  # GK
  "Goalkeeper": "GK",
  # CB
  "Center Back": "CB",
  "Left Center Back": "CB",
  "Right Center Back": "CB",
  # FB
  "Left Back": "FB",
  "Right Back": "FB",
  "Left Wing Back": "FB",
  "Right Wing Back": "FB",
  # CM
  "Center Defensive Midfield": "CM",
  "Left Defensive Midfield": "CM",
  "Right Defensive Midfield": "CM",
  "Center Midfield": "CM",
  "Left Center Midfield": "CM",
  "Right Center Midfield": "CM",
  "Left Midfield": "CM",
  "Right Midfield": "CM",
  # AM
  "Attacking Midfield": "AM",
  "Center Attacking Midfield": "AM",
  "Left Attacking Midfield": "AM",
  "Right Attacking Midfield": "AM",
  "Left Wing": "AM",
  "Right Wing": "AM",
  "Secondary Striker": "AM",
  # FW
  "Center Forward": "FW",
  "Striker": "FW",
  "Left Center Forward": "FW",
  "Right Center Forward": "FW",
}


def map_position_group(position_name):
    if position_name is None or (isinstance(position_name, float) and np.isnan(position_name)):
        return np.nan
    name = str(position_name).strip()
    if name in ("", "Unknown"):
        return np.nan
    return POSITION_MAP.get(name, np.nan)


def cramers_v(chi2, n, r, k):
    """Effect size for chi-square on r×k contingency table."""
    if n == 0:
        return np.nan
    return float(np.sqrt(chi2 / (n * min(r - 1, k - 1))))


def row_pct_table(ct):
    """Row-normalized percentages (each row sums to 100)."""
    return ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)


def plot_heatmap(pct_df, title, xlabel="Cluster", ylabel="Position", figsize=(10, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        pct_df,
        annot=True,
        fmt=".1f",
        cmap="YlOrRd",
        linewidths=0.5,
        cbar_kws={"label": "% of row"},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    plt.show()

print(f"Mapped {len(POSITION_MAP)} StatsBomb positions → 6 groups")

---
## 2. Load and join data

Grain: **player × competition × season** for season stats and positions.  
Primary position = mode of `position_name` weighted by minutes in that competition-season.

`cluster_assignments` is joined on `(player_id, competition_id, season_id)` when those columns exist; otherwise the cell falls back to a deduped player-level join. Source tables use **`dbt_intermediate`** (PR #124), aligned with `src/ml/cluster.py`.

In [ ]:
def _sql_strings(values):
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)


top5_names_sql = _sql_strings(TOP5_RAW_NAMES)

print("=== Competitions in int_player_season_stats (top 20 by rows) ===")
comp_diag = spark.sql(
    f"""
    SELECT competition_id, competition_name, COUNT(*) AS n_rows
    FROM {INT_SEASON}
    GROUP BY 1, 2
    ORDER BY n_rows DESC
    LIMIT 20
    """
).toPandas()
display(comp_diag)

in_top5 = comp_diag[comp_diag["competition_name"].isin(TOP5_RAW_NAMES)]
if in_top5.empty:
    print("WARNING: none of TOP5_RAW_NAMES appear in season stats — check ingestion coverage.")
else:
    missing = set(COMP_LABEL_ORDER) - set(
        in_top5["competition_name"].map(NAME_TO_LABEL).unique()
    )
    if missing:
        print(f"Leagues with 0 rows after name filter (may be missing from pipeline): {sorted(missing)}")


def _table_columns(table_fqn):
    rows = spark.sql(f"DESCRIBE TABLE {table_fqn}").collect()
    return {
        r.col_name.lower()
        for r in rows
        if r.col_name and not str(r.col_name).startswith("#")
    }


cluster_cols = _table_columns(CLUSTER_TABLE)
join_keys = [k for k in ("player_id", "competition_id", "season_id") if k in cluster_cols]
if "player_id" not in join_keys:
    raise ValueError(f"{CLUSTER_TABLE} has no player_id column: {sorted(cluster_cols)}")

if join_keys == ["player_id"]:
    print(
        "Note: cluster_assignments is player-level only (no competition_id/season_id). "
        "Joining on player_id; one cluster row per player (max total_minutes if available)."
    )
else:
    print(f"Joining clusters on: {', '.join(join_keys)}")

if "cluster" in cluster_cols:
    cluster_expr = "c.cluster"
elif "cluster_id" in cluster_cols:
    cluster_expr = "c.cluster_id AS cluster"
else:
    raise ValueError(f"No cluster column in {CLUSTER_TABLE}: {sorted(cluster_cols)}")

if "archetype" in cluster_cols:
    archetype_expr = "c.archetype"
elif "cluster_label" in cluster_cols:
    archetype_expr = "c.cluster_label AS archetype"
else:
    archetype_expr = "CAST(NULL AS STRING) AS archetype"

pc1_expr = "c.pc1" if "pc1" in cluster_cols else "CAST(NULL AS DOUBLE) AS pc1"
pc2_expr = "c.pc2" if "pc2" in cluster_cols else "CAST(NULL AS DOUBLE) AS pc2"

join_on = " AND ".join(f"s.{k} = c.{k}" for k in join_keys)

if join_keys == ["player_id"]:
    order_clause = (
        "c.total_minutes DESC NULLS LAST"
        if "total_minutes" in cluster_cols
        else "1"
    )
    cluster_source = f"""
clusters AS (
  SELECT * EXCEPT (rn)
  FROM (
    SELECT c.*, ROW_NUMBER() OVER (PARTITION BY c.player_id ORDER BY {order_clause}) AS rn
    FROM {CLUSTER_TABLE} c
  )
  WHERE rn = 1
)
"""
    cluster_ref = "clusters c"
else:
    cluster_source = ""
    cluster_ref = f"{CLUSTER_TABLE} c"

load_sql = f"""
WITH club_matches AS (
  SELECT match_id, competition_id, season_id
  FROM {MATCHES}
  WHERE COALESCE(is_international, false) = false
    AND competition_name IN ({top5_names_sql})
),
pos_by_match AS (
  SELECT
    pms.player_id,
    cm.competition_id,
    cm.season_id,
    pms.position_name,
    SUM(pms.minutes_played) AS minutes_played
  FROM {INT_MATCH} pms
  INNER JOIN club_matches cm ON pms.match_id = cm.match_id
  WHERE pms.position_name IS NOT NULL
    AND pms.position_name NOT IN ('Unknown', '')
  GROUP BY 1, 2, 3, 4
),
primary_position AS (
  SELECT
    player_id,
    competition_id,
    season_id,
    position_name AS primary_position
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY player_id, competition_id, season_id
        ORDER BY minutes_played DESC
      ) AS rn
    FROM pos_by_match
  )
  WHERE rn = 1
),
{cluster_source}
SELECT
  s.player_id,
  s.player_name,
  s.competition_id,
  s.competition_name,
  s.season_id,
  s.season_name,
  s.total_minutes,
  pp.primary_position,
  {cluster_expr},
  {archetype_expr},
  {pc1_expr},
  {pc2_expr}
FROM {INT_SEASON} s
INNER JOIN primary_position pp
  ON s.player_id = pp.player_id
 AND s.competition_id = pp.competition_id
 AND s.season_id = pp.season_id
INNER JOIN {cluster_ref}
  ON {join_on}
WHERE s.competition_name IN ({top5_names_sql})
"""

raw = spark.sql(load_sql).toPandas()
raw["competition_label"] = raw["competition_name"].map(NAME_TO_LABEL)
raw["position_group"] = raw["primary_position"].map(map_position_group)

unmapped = raw[raw["position_group"].isna()]["primary_position"].dropna().unique()
if len(unmapped):
    print("Unmapped StatsBomb positions (review POSITION_MAP):")
    print(sorted(unmapped))

df = raw.dropna(subset=["position_group", "cluster"]).copy()
df["cluster"] = df["cluster"].astype(int)
df["position_group"] = pd.Categorical(df["position_group"], categories=POSITION_ORDER, ordered=True)
df["competition_label"] = pd.Categorical(
    df["competition_label"], categories=COMP_LABEL_ORDER, ordered=True
)

n_clusters = df["cluster"].nunique()
cluster_ids = sorted(df["cluster"].unique())

print(f"Rows loaded (top 5 leagues, with position + cluster): {len(df):,}")
print(f"Player-seasons: {len(df):,} | Players: {df['player_id'].nunique():,}")
print(f"Clusters: {n_clusters} | Position groups: {df['position_group'].nunique()}")
print("\nRows by competition:")
print(df["competition_label"].value_counts().sort_index().to_string())
print("\nRows by position group:")
print(df["position_group"].value_counts().sort_index().to_string())

display(df.head(10))

---
## 3. Key Question 1 — Does position predict cluster membership?

**Test:** χ² independence on `position_group` × `cluster`.  
**Effect size:** Cramér's V (0 ≈ no association, 1 ≈ strong association).  
**Visualization:** Row-normalized heatmap — for each position, % of players in each cluster.

In [ ]:
ct_pos_cluster = pd.crosstab(df["position_group"], df["cluster"])
ct_pos_cluster = ct_pos_cluster.reindex(index=POSITION_ORDER, columns=cluster_ids, fill_value=0)

pct_pos_cluster = row_pct_table(ct_pos_cluster)

summary_kq1_counts = ct_pos_cluster.copy()
summary_kq1_counts["total"] = summary_kq1_counts.sum(axis=1)

print("=== KQ1: Position × cluster counts ===")
display(summary_kq1_counts)

print("=== KQ1: Position × cluster row % ===")
display(pct_pos_cluster)

chi2_kq1, p_kq1, dof_kq1, _ = chi2_contingency(ct_pos_cluster.values)
v_kq1 = cramers_v(chi2_kq1, ct_pos_cluster.values.sum(), *ct_pos_cluster.shape)

print(f"Chi-square: χ² = {chi2_kq1:.2f}, df = {dof_kq1}, p-value = {p_kq1:.2e}")
print(f"Cramér's V = {v_kq1:.3f}")
if p_kq1 < 0.05:
    print("→ Position and cluster are statistically associated (reject independence at α=0.05).")
else:
    print("→ No significant association at α=0.05 (position may not predict cluster well).")

In [ ]:
plot_heatmap(
    pct_pos_cluster,
    title="KQ1: Cluster mix within each position (row %)",
    xlabel="Cluster ID",
    ylabel="Position group",
    figsize=(11, 5),
)

# Dominant cluster per position (for report narrative)
dominant = pct_pos_cluster.idxmax(axis=1).rename("dominant_cluster")
dominant_pct = pct_pos_cluster.max(axis=1).rename("dominant_cluster_pct")
kq1_dominant = pd.concat([dominant, dominant_pct], axis=1)
print("Dominant cluster per position:")
display(kq1_dominant)

---
## 4. Key Question 2 — Does competition affect role distribution?

We test two views:
1. **Overall cluster mix** differs across leagues (competition × cluster).
2. **Position-specific role profiles** — position × cluster heatmap **per competition** (visual comparison).

In [ ]:
ct_comp_cluster = pd.crosstab(df["competition_label"], df["cluster"])
ct_comp_cluster = ct_comp_cluster.reindex(index=list(COMP_LABEL_ORDER), columns=cluster_ids, fill_value=0)
pct_comp_cluster = row_pct_table(ct_comp_cluster)

print("=== KQ2a: Competition × cluster counts ===")
display(ct_comp_cluster)
print("=== KQ2a: Competition × cluster row % ===")
display(pct_comp_cluster)

chi2_kq2, p_kq2, dof_kq2, _ = chi2_contingency(ct_comp_cluster.values)
v_kq2 = cramers_v(chi2_kq2, ct_comp_cluster.values.sum(), *ct_comp_cluster.shape)

print(f"Chi-square (competition × cluster): χ² = {chi2_kq2:.2f}, df = {dof_kq2}, p = {p_kq2:.2e}")
print(f"Cramér's V = {v_kq2:.3f}")

In [ ]:
plot_heatmap(
    pct_comp_cluster,
    title="KQ2a: Cluster mix by competition (row %)",
    xlabel="Cluster ID",
    ylabel="Competition",
    figsize=(11, 4),
)

In [ ]:
# KQ2b: position × cluster within each competition
pct_by_comp = {}
v_by_comp = {}

for comp in COMP_LABEL_ORDER:
    sub = df[df["competition_label"] == comp]
    ct = pd.crosstab(sub["position_group"], sub["cluster"])
    ct = ct.reindex(index=POSITION_ORDER, columns=cluster_ids, fill_value=0)
    pct = row_pct_table(ct)
    pct_by_comp[comp] = pct
    if ct.values.sum() > 0 and ct.shape[0] > 1 and ct.shape[1] > 1:
        chi2_c, _, _, _ = chi2_contingency(ct.values)
        v_by_comp[comp] = cramers_v(chi2_c, ct.values.sum(), *ct.shape)
    else:
        v_by_comp[comp] = np.nan

kq2_position_effect = pd.Series(v_by_comp, name="cramers_v_position_x_cluster").sort_values(ascending=False)
print("Cramér's V (position × cluster) within each competition:")
display(kq2_position_effect.to_frame())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax, comp in zip(axes, COMP_LABEL_ORDER):
    sns.heatmap(
        pct_by_comp[comp],
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        linewidths=0.3,
        cbar=False,
        ax=ax,
    )
    ax.set_title(comp)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Position")
axes[-1].axis("off")
fig.suptitle("KQ2b: Position × cluster (row %) by competition", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Difference from pooled position×cluster profile (highlights league-specific shifts)
pooled_pct = pct_pos_cluster.copy()
delta_rows = []
for comp in COMP_LABEL_ORDER:
    delta = pct_by_comp[comp] - pooled_pct
    delta_rows.append(delta.stack().rename(comp))
delta_long = pd.concat(delta_rows, axis=1)
delta_long.index.names = ["position_group", "cluster"]

# Wide table: max absolute deviation per competition (summary for report)
max_delta = delta_long.abs().groupby(level=0).max().max(axis=1)
max_delta = max_delta.reindex(POSITION_ORDER).rename("max_abs_pct_point_deviation")
print("Positions with largest league-specific deviation from pooled cluster mix:")
display(max_delta.sort_values(ascending=False).to_frame())

---
## 5. Report summary (copy to final report § RQ1)

Run all cells above, then use the printed statistics and tables in the Quarto report.

In [ ]:
report_summary = pd.DataFrame(
    {
        "metric": [
            "player_season_rows",
            "distinct_players",
            "n_clusters",
            "kq1_chi2",
            "kq1_p_value",
            "kq1_cramers_v",
            "kq2_competition_chi2",
            "kq2_competition_p_value",
            "kq2_competition_cramers_v",
        ],
        "value": [
            len(df),
            df["player_id"].nunique(),
            n_clusters,
            round(chi2_kq1, 2),
            f"{p_kq1:.2e}",
            round(v_kq1, 3),
            round(chi2_kq2, 2),
            f"{p_kq2:.2e}",
            round(v_kq2, 3),
        ],
    }
)

# Databricks `display()` may convert pandas->Spark via Arrow.
# Mixed dtypes in a column can trigger Arrow conversion errors, so we stringify.
report_summary["value"] = report_summary["value"].astype(str)

print("=== RQ1 executive summary ===")
display(report_summary)

interpretation = []
if p_kq1 < 0.05:
    interpretation.append(
        f"Position predicts cluster membership (χ² p={p_kq1:.2e}, Cramér's V={v_kq1:.3f})."
    )
else:
    interpretation.append(
        f"Position does not significantly predict cluster membership at α=0.05 (p={p_kq1:.2e})."
    )
if p_kq2 < 0.05:
    interpretation.append(
        f"Competition affects overall cluster mix (χ² p={p_kq2:.2e}, Cramér's V={v_kq2:.3f})."
    )
else:
    interpretation.append(
        f"Overall cluster mix is similar across top-five leagues (p={p_kq2:.2e})."
    )
interpretation.append(
    "Compare KQ2b faceted heatmaps and per-league Cramér's V for position-specific tactical differences."
)

print("\nSuggested narrative bullets:")
for line in interpretation:
    print(f"  • {line}")